## RAG using only HuggingFace Embedding & Language Models

In [ ]:
# Both PyPDFLoader and PyPDFDirectoryLoader utilize the PyPDF library to read and extract text from PDF documents
from langchain_community.document_loaders import PyPDFLoader            # used to load a single PDF file
from langchain_community.document_loaders import PyPDFDirectoryLoader   # used to load all PDF files from a specified directory
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_core.prompts import PromptTemplate                       
from langchain_classic.chains import RetrievalQA                        # used to create a question-answering chain that retrieves relevant information from the vector store based on user queries

c:\All_Work\LangChainProj\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
## Reading the PDFs from the folder

loader=PyPDFDirectoryLoader("./cricket_docs")               # loading all PDF files at once

documents=loader.load()      # documents is a list of Document objects, where each Document contains the text content and metadata (like source) of a PDF file

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

final_documents=text_splitter.split_documents(documents)    # splitting the text content of each Document into smaller chunks
final_documents[0]                                          # the first chunk of text from the first Document to verify the splitting process

Document(metadata={'producer': 'http://createpdf.adobe.com V5.1', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2003-09-22T22:07:12-07:00', 'moddate': '2003-09-22T22:07:12-07:00', 'author': 'www', 'title': 'Microsoft Word - 3F6FD898-3023-08015E.doc', 'source': 'cricket_docs\\intro.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='The Game of Cricket \n \n          B a t s m e n  \n \n \nBowler           Wickets Umpire   Crease \n \n Cricket field: The rectangular patch in the center is called the pitch.               Pitch \n \n \n \n \n \nA typical Cricket game has 11 players in each team. It is fundamentally very similar \nto baseball. It is played with a bat and a ball. The center of the field is a rectangular area of \n22 meters called a pitch. In cricket there are only two bases (called creases) on either ends \nof the pitch. Three wooden sticks called wickets or stumps are placed inside each crease. \nA batsman stands on the pitch in front of the wick

In [3]:
len(final_documents)

118

## Huggingface Models

### Embedding model

In [ ]:


huggingface_embeddings=HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",            # this is a specific model from Hugging Face that is designed for generating sentence embeddings. It is a smaller version of the BGE model, optimized for efficiency
    model_kwargs={'device':'cpu'},                  # since the pdfs are very small, we can use CPU for embedding generation. For larger documents, using GPU would be more efficient
    encode_kwargs={'normalize_embeddings':True}     # normalizing the embeddings to have unit length, which can improve the performance of similarity search in the vector store
)

C:\Users\nisha\AppData\Local\Temp\ipykernel_14984\1638020884.py:1: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings=HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8736.40it/s]


In [ ]:
import numpy as np
print(np.array(huggingface_embeddings.embed_query(final_documents[0].page_content)))        # generating the embedding for the first chunk of text from the first Document and printing its shape to verify that the embedding generation process is working correctly
print(np.array(huggingface_embeddings.embed_query(final_documents[0].page_content)).shape)  # the shape of the embedding will depend on the specific model used, but it typically will be a 1D array with a length corresponding to the dimensionality of the embedding space (e.g., as seen below 384 for the all-MiniLM-l6-v2 model)

[ 1.06053764e-03  2.30884422e-02 -5.96141443e-02 -4.97971475e-02
 -3.10244132e-03  5.03003746e-02  5.60233481e-02  7.98199885e-03
  6.10376056e-03  8.41159970e-02  4.63064387e-02 -9.08539891e-02
  8.69710464e-03  1.50126182e-02 -2.09513810e-02  1.67001393e-02
 -4.10409085e-02 -3.39567028e-02  4.32892889e-02  3.40002850e-02
  8.65461454e-02  7.96492547e-02 -9.23282560e-03 -6.22412562e-03
 -5.53224096e-03  2.94490773e-02 -4.43008132e-02 -9.53942835e-02
  6.69398019e-03 -1.41680509e-01  4.03019451e-02  6.68125823e-02
 -2.79091857e-02  4.74415235e-02 -3.84996529e-03 -7.03136921e-02
 -3.73171270e-02  4.46378402e-02  1.48579059e-02 -3.09976190e-02
  1.69478767e-02  6.86543956e-02  7.85727575e-02 -1.92861687e-02
 -4.44362015e-02  9.61626414e-03 -3.71062271e-02  3.71321738e-02
  4.12052162e-02 -3.17257158e-02  1.39513081e-02 -9.41670779e-03
 -7.97349401e-03 -9.93126631e-03 -3.20066959e-02 -4.71530072e-02
  5.97642064e-02  2.77422220e-02  4.64041298e-03  1.88191049e-02
  5.66288680e-02  5.64755

In [ ]:
## VectorStore Creation

vectorstore=FAISS.from_documents(final_documents[:60],huggingface_embeddings)  # creating a FAISS vector store from the first 60 chunks of text out of the total 118 chunks with dimensionality of 384 (for the BGE model)

In [7]:
## Query using Similarity Search

query="WHAT IS CRICKET?"
relevant_docments=vectorstore.similarity_search(query)

print(relevant_docments[0].page_content)

The Game of Cricket 
 
          B a t s m e n  
 
 
Bowler           Wickets Umpire   Crease 
 
 Cricket field: The rectangular patch in the center is called the pitch.               Pitch 
 
 
 
 
 
A typical Cricket game has 11 players in each team. It is fundamentally very similar 
to baseball. It is played with a bat and a ball. The center of the field is a rectangular area of 
22 meters called a pitch. In cricket there are only two bases (called creases) on either ends 
of the pitch. Three wooden sticks called wickets or stumps are placed inside each crease. 
A batsman stands on the pitch in front of the wickets in one crease, and tries to hit the ball 
that a bowler (pitcher) throws towards him. His partner stands in the other crease waiting 
for his turn. Aim of the batsman is to hit the ball to make as many runs (points) as he can, 
while the aim of the bowler is to get the batsman out while giving away the least runs 
possible.


In [ ]:
## if we want to get more relevant documents, we can specify the number of documents to retrieve using the "k".

retriever=vectorstore.as_retriever(search_type="similarity",search_kwargs={"k":5})  # converting the vector store into a retriever (an interface connected to a vectorstore) that can be used in a RetrievalQA chain

tags=['FAISS', 'HuggingFaceBgeEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001CF531BF100> search_kwargs={'k': 5}


### HuggingFace LLM

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

huggingface_api_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if huggingface_api_key is None:
    raise ValueError("HUGGINGFACEHUB_API_TOKEN is not set. Please check your .env file.")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = huggingface_api_key


In [ ]:
## Huggingface server LLMs - using ChatHuggingFace with Llama 3.1 (works with free HF Inference API on server, i.e. no need to download locally)

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",     # this model is available on the free HF Inference API
    temperature=0.1,
    max_new_tokens=500,                             # this limit ensures that the model generates a concise response, instead of a overly verbose one
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
)

hf = ChatHuggingFace(llm=llm)                       # creating a ChatHuggingFace instance that wraps the Hugging Face endpoint, allowing us to use it as a language model in our RetrievalQA chain

query = "What is cricket?"
response = hf.invoke(query)                         # invoke() - sends the query to the Hugging Face model and retrieves the generated response based on the input query    
print(response.content)

Cricket is a popular team sport played with a bat and ball on a rectangular field with two sets of three stumps (wickets) at each end. The game originated in England in the 16th century and is now played in many countries around the world, particularly in the Indian subcontinent, Australia, the Caribbean, and the United Kingdom.

Here's a brief overview of the game:

**Objective:** The objective of cricket is to score runs by hitting the ball with a bat and running between the wickets, while the opposing team tries to stop them by getting the batsmen out.

**Gameplay:**

1. Two teams of 11 players each take the field.
2. One team bats, while the other team fields.
3. The batting team sends two batsmen onto the field, who take turns hitting the ball bowled by the opposing team's bowler.
4. The batsmen score runs by hitting the ball and running between the wickets.
5. The fielding team tries to get the batsmen out by catching the ball, hitting the stumps, or running the batsmen out.
6. T

In [11]:
#Hugging Face models can also be run locally through the HuggingFacePipeline class

'''
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}
)

llm = hf 
llm.invoke(query)
'''

'\nfrom langchain_community.llms.huggingface_pipeline import HuggingFacePipeline\n\nhf = HuggingFacePipeline.from_model_id(\n    model_id="meta-llama/Llama-3.1-8B-Instruct",\n    task="text-generation",\n    pipeline_kwargs={"temperature": 0, "max_new_tokens": 300}\n)\n\nllm = hf \nllm.invoke(query)\n'

In [ ]:
prompt_template="""
Use the following piece of context to answer the question asked.
Provide the answer only based on the context.
{context}                           # a placeholder for the retrieved context from the vector store, which will be dynamically filled in when the RetrievalQA chain is executed
Question:{question}                 # a placeholder for the user query, which will also be dynamically filled in during execution

Helpful Answers:
 """

In [13]:
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])

In [ ]:
## RetrievalQA Chain Creation

# creating a RetrievalQA chain using the ChatHuggingFace instance as the language model, the custom prompt template, and the retriever connected to the FAISS vector store.
retrievalQA=RetrievalQA.from_chain_type(    
    llm=hf,                                 # the ChatHuggingFace instance that wraps the Hugging Face endpoint, allowing us to use it as a language model in our RetrievalQA chain
    chain_type="stuff",                     # "stuff" is a simple chain type that takes all retrieved documents and stuffs them into the prompt for the language model. This is suitable for small contexts, & for larger contexts, other chain types like "map_reduce" or "refine" are more effective
    retriever=retriever,                    # the retriever connected to the FAISS vector store, which will retrieve relevant documents based on user queries
    return_source_documents=True,           # to know from where the answer is coming up (inlcudes the source document for the output)
    chain_type_kwargs={"prompt":prompt}     # passing the custom prompt template to the RetrievalQA chain, which will be used to format the input for the language model based on the retrieved context and user query
)

In [15]:
query="""Cricket rules"""

In [ ]:
## calling the QA chain with our query

result = retrievalQA.invoke({"query": query})
print(result['result'])

Based on the provided context, the following are the cricket rules:

1. **Law 15: Basic Rules**
   - The team list must be registered before the game and cannot be changed unless there is an injury/illness, and agreed to by the opposing coach/captain.
   - The value of the boundaries shall be as in regulation Cricket.
   - The ball to be used will be decided by the Organizing/Tournament Committee.
   - Players must wear their standard cricket uniform, long trousers, proper sports footwear, and cricket pads.

2. **Law 7: Scoring**
   - Runs may be scored as follows:
     - Every time the batters cross between the batting crease and the running creases while the ball is in play, 1 run is scored.
     - Overthrows are counted as runs.
     - When the ball reaches or is hit over the agreed boundaries, 4 and 6 runs respectively are scored.
     - Two bonus runs are scored for a no-ball or wide.

3. **Law 8: Wides**
   - A wide shall be called if, as the ball crosses the batting crease, it i